In [4]:
"""AI-generated (Claude Code): checks our bots against the upgraded ai-poker-server-master.

The TA's server upgrade added rules and match settings that bots have to follow:
  - config.toml: 5 second turn timeout, max_hands = 100 per match
  - fold/check/call must not carry an amount
  - no raising when every opponent is already all-in
  - a short all-in does not reopen raising for a player who already acted
  - the server re-sends state after a rejected action, so a client must not answer the same turn twice

Part 1 plays full matches on the server's engine (no network).
Part 2 starts the real FastAPI server and connects the bots through run_bot.play over websockets.

Run every cell; the last cell runs the tests with pytest.
"""

import asyncio
import contextlib
import io
import json
import os
import random
import socket
import sys
import threading
import time

ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "models")):
    parent = os.path.dirname(ROOT)
    assert parent != ROOT, "run this from inside the project folder"
    ROOT = parent
sys.path.insert(0, ROOT)
sys.path.append(os.path.join(ROOT, "ai-poker-server-master"))

import pytest
from app.config import load_config
from app.engine import PokerEngine
from models.bot.AllInBot import AllInBot
from models.bot.BaseBot import BaseBot
from models.bot.MelBot import MelBot
from models.bot.RandomBot import RandomBot

In [5]:
SETTINGS = load_config()  # the same config.toml the server loads
ALL_BOTS = [MelBot, BaseBot, AllInBot, RandomBot, RandomBot]


def new_engine():
    return PokerEngine(
        starting_stack=SETTINGS.starting_stack,
        small_blind=SETTINGS.small_blind,
        big_blind=SETTINGS.big_blind,
        max_hands=SETTINGS.max_hands,
    )


def seat(engine, classes):
    bots = {}
    for i, cls in enumerate(classes):
        pid = f"{cls.__name__}-{i}"
        bots[pid] = cls()
        engine.add_player(pid, pid)
    return bots


def play_match(engine, bots):
    """plays hands until the server would end the match; returns every rejected action"""
    rejected = []
    while engine.can_start_hand():
        engine.start_hand()
        while engine.phase not in ("WAITING", "SHOWDOWN") and engine.current_turn:
            pid = engine.current_turn
            state = engine.snapshot(pid)
            action, amount = bots[pid].act(state, pid)
            result = engine.place_action(pid, action, amount)
            if not result["ok"]:
                rejected.append((pid, action, amount, result["error"]))
                engine.place_action(
                    pid,
                    (
                        "check"
                        if engine.players[pid].bet == engine.current_bet
                        else "call"
                    ),
                    0,
                )
        if engine.phase == "SHOWDOWN":
            engine.finalize_showdown()
    return rejected


# ---------------------------------------------------------------- Part 1: engine rules


@pytest.mark.parametrize("seed", range(40))
def test_full_match_no_bot_action_is_rejected(seed):
    """a whole match with the real config: chips are conserved and the engine accepts every action"""
    random.seed(seed)
    engine = new_engine()
    bots = seat(engine, ALL_BOTS)
    rejected = play_match(engine, bots)
    assert rejected == []
    assert engine.match_complete
    assert engine.hands_completed <= SETTINGS.max_hands
    assert sum(
        p.stack for p in engine.players.values()
    ) == SETTINGS.starting_stack * len(bots)
    assert sum(p.profit_loss for p in engine.players.values()) == 0


@pytest.mark.parametrize("cls", [MelBot, AllInBot, RandomBot, BaseBot])
@pytest.mark.parametrize("seed", range(10))
def test_heads_up_match_no_rejections(cls, seed):
    """heads-up hits the all-in and short-stack rules far more often than a full table"""
    random.seed(seed)
    engine = new_engine()
    bots = seat(engine, [cls, AllInBot])
    assert play_match(engine, bots) == []


def test_match_stops_at_max_hands():
    """check/call bots never bust each other, so the match must end on the hand limit"""
    engine = new_engine()
    seat(engine, [BaseBot, BaseBot])
    bots = {pid: BaseBot() for pid in engine.players}
    play_match(engine, bots)
    assert engine.hands_completed == SETTINGS.max_hands
    assert not engine.can_start_hand()


def test_snapshot_still_has_every_field_the_bots_read():
    engine = new_engine()
    bots = seat(engine, [MelBot, BaseBot])
    engine.start_hand()
    state = engine.snapshot(engine.current_turn)
    for key in (
        "players",
        "community_cards",
        "pot",
        "big_blind",
        "current_turn",
        "hand_number",
        "phase",
        "current_bet",
        "minimum_raise",
    ):
        assert key in state
    for key in ("id", "stack", "bet", "folded", "all_in", "connected", "hole_cards"):
        assert key in state["players"][0]
    me = next(p for p in state["players"] if p["id"] == engine.current_turn)
    assert all(
        isinstance(c["rank"], int) and isinstance(c["suit"], str)
        for c in me["hole_cards"]
    )
    bots[engine.current_turn].act(
        state, engine.current_turn
    )  # parses the cards without errors


def _all_in_spot():
    """3 players: two shove preflop, the third (MelBot holding aces) is left facing only all-in players"""
    engine = new_engine()
    for pid in ("a", "b", "mel"):
        engine.add_player(pid, pid)
    engine.start_hand()
    engine.place_action(
        "a", "raise", engine.players["a"].bet + engine.players["a"].stack
    )
    engine.place_action(
        "b", "call", 0
    )  # equal stacks, so calling the shove puts b all-in too
    assert engine.current_turn == "mel"
    engine.players["mel"].hole_cards = [
        c for c in engine._build_deck() if c.rank == 14
    ][:2]
    return engine


@pytest.mark.parametrize("cls", [MelBot, AllInBot, RandomBot])
def test_never_raises_into_only_all_in_players(cls):
    engine = _all_in_spot()
    state = engine.snapshot("mel")
    bot = cls()
    for _ in range(20):  # RandomBot is random, try it several times
        action, amount = bot.act(state, "mel")
        assert action in ("call", "fold"), action
        assert engine.place_action("mel", action, amount)["ok"] or action == "fold"
        if engine.current_turn != "mel":
            break


def test_short_all_in_does_not_reopen_raising():
    """mel raises, a short stack moves all-in for slightly more, mel may only call or fold"""
    engine = new_engine()
    for pid in ("mel", "short", "big"):
        engine.add_player(pid, pid)
    engine.start_hand()
    engine.players["short"].stack = (
        90 - engine.players["short"].bet
    )  # all-in total of 90
    mel = AllInBot.__new__(AllInBot)
    BaseBot.__init__(mel)
    # walk to mel's turn with calls, then make mel raise to 80
    while engine.current_turn != "mel":
        engine.place_action(engine.current_turn, "call", 0)
    state = engine.snapshot("mel")
    mel.update_from_state(state, "mel")
    mel._last_turn = (state["hand_number"], state["phase"], 80)
    assert engine.place_action("mel", "raise", 80)["ok"]
    while engine.current_turn != "mel":
        pid = engine.current_turn
        if pid == "short":
            assert engine.place_action(pid, "raise", 90)[
                "ok"
            ]  # short all-in, raise of 10 < minimum 60
        else:
            engine.place_action(pid, "call", 0)
    state = engine.snapshot("mel")
    mel.update_from_state(state, "mel")
    assert not mel.can_raise(state)
    assert mel.choose_action(state) == ("call", 0)


def test_passive_actions_never_carry_an_amount():
    random.seed(3)
    engine = new_engine()
    bots = seat(engine, ALL_BOTS)
    seen = 0
    while engine.can_start_hand() and seen < 2000:
        engine.start_hand()
        while engine.phase not in ("WAITING", "SHOWDOWN") and engine.current_turn:
            pid = engine.current_turn
            action, amount = bots[pid].act(engine.snapshot(pid), pid)
            if action in ("fold", "check", "call"):
                assert amount == 0
            assert engine.place_action(pid, action, amount)["ok"]
            seen += 1
        if engine.phase == "SHOWDOWN":
            engine.finalize_showdown()


def test_melbot_decides_well_inside_turn_timeout():
    """the server folds a bot that takes longer than turn_timeout_seconds"""
    random.seed(5)
    engine = new_engine()
    bots = seat(engine, ALL_BOTS)
    worst = 0.0
    while engine.can_start_hand():
        engine.start_hand()
        while engine.phase not in ("WAITING", "SHOWDOWN") and engine.current_turn:
            pid = engine.current_turn
            state = engine.snapshot(pid)
            start = time.perf_counter()
            action, amount = bots[pid].act(state, pid)
            worst = max(worst, time.perf_counter() - start)
            engine.place_action(pid, action, amount)
        if engine.phase == "SHOWDOWN":
            engine.finalize_showdown()
    assert (
        worst < SETTINGS.turn_timeout_seconds / 10
    ), f"slowest decision took {worst:.3f}s"


# ---------------------------------------------------------------- Part 2: live server over websockets

websockets = pytest.importorskip("websockets")
uvicorn = pytest.importorskip("uvicorn")


@pytest.fixture(scope="module")
def live_server():
    """the real server app on a free local port, with no showdown pause so a match takes seconds"""
    from app import server

    sock = socket.socket()
    sock.bind(("127.0.0.1", 0))
    port = sock.getsockname()[1]
    sock.close()
    uv = uvicorn.Server(
        uvicorn.Config(server.app, host="127.0.0.1", port=port, log_level="warning")
    )
    thread = threading.Thread(target=uv.run, daemon=True)
    thread.start()
    deadline = time.time() + 10
    while not uv.started:
        assert time.time() < deadline, "server did not start"
        time.sleep(0.05)
    yield server, f"ws://127.0.0.1:{port}/ws"
    uv.should_exit = True
    thread.join(5)


def reset_server(server):
    server.engine = new_engine()
    server.connections.clear()
    server.spectators.clear()
    server.SHOWDOWN_REVEAL_SECONDS = 0.0
    server.ACTION_DELAY_SECONDS = 0.0
    server.MIN_HAND_GAP_SECONDS = 0.01
    server.TURN_TIMEOUT_SECONDS = SETTINGS.turn_timeout_seconds
    server.paused = False
    server.running = False


async def run_live_match(url, bots, timeout=60):
    """connects every bot with run_bot.play, presses Start as the instructor, waits for the match to end.
    returns the final state, the bots' printed output, and every turn the server timed out
    """
    import run_bot

    out = io.StringIO()
    timeouts = []
    with contextlib.redirect_stdout(out):
        tasks = [
            asyncio.create_task(run_bot.play(bot, pid, pid, url))
            for pid, bot in bots.items()
        ]
        try:
            async with websockets.connect(url) as instructor:
                await instructor.send(json.dumps({"type": "observe"}))
                while True:  # wait until every bot is seated
                    msg = json.loads(await asyncio.wait_for(instructor.recv(), 10))
                    if msg["type"] == "state" and len(msg["payload"]["players"]) == len(
                        bots
                    ):
                        break
                await instructor.send(json.dumps({"type": "resume"}))
                end = time.time() + timeout
                while True:
                    msg = json.loads(
                        await asyncio.wait_for(
                            instructor.recv(), max(0.1, end - time.time())
                        )
                    )
                    if msg["type"] != "state":
                        continue
                    state = msg["payload"]
                    if (state.get("last_action") or {}).get("timeout"):
                        timeouts.append(
                            (state["hand_number"], state["last_action"]["timeout"])
                        )
                    if state["match_complete"]:
                        break
            for task in tasks:
                assert (
                    not task.done()
                ), task.exception()  # a crashed bot would be folded by the timeout
        finally:
            for task in tasks:
                task.cancel()
            await asyncio.gather(*tasks, return_exceptions=True)
    return state, out.getvalue(), timeouts


@pytest.mark.parametrize("seed", range(3))
def test_live_match_through_run_bot(live_server, seed):
    server, url = live_server
    reset_server(server)
    random.seed(seed)
    bots = {f"{cls.__name__}-{i}": cls() for i, cls in enumerate(ALL_BOTS)}
    state, log, timeouts = asyncio.run(run_live_match(url, bots))
    assert state["match_complete"]
    assert 0 < state["hands_completed"] <= SETTINGS.max_hands
    assert sum(p["stack"] for p in state["players"]) == SETTINGS.starting_stack * len(
        bots
    )
    assert timeouts == []
    rejected = [
        line
        for line in log.splitlines()
        if line.startswith(("Rejected", "Server error"))
    ]
    assert rejected == []


class IllegalRaiseBot(BaseBot):
    """always asks for a raise bigger than its stack, which the server must reject"""

    def choose_action(self, state):
        return "raise", self.current_bet + self.chips + 1000


def test_live_rejected_action_falls_back_once(live_server):
    """after a rejection run_bot sends one check/call; the server's repeated state must not trigger a second
    action (stray actions would show up as 'not this player's turn' or showdown errors)
    """
    server, url = live_server
    reset_server(server)
    server.engine = PokerEngine(
        starting_stack=SETTINGS.starting_stack,
        small_blind=SETTINGS.small_blind,
        big_blind=SETTINGS.big_blind,
        max_hands=5,
    )
    bots = {"illegal": IllegalRaiseBot(), "base": BaseBot()}
    state, log, timeouts = asyncio.run(run_live_match(url, bots))
    assert state["hands_completed"] == 5
    assert timeouts == []
    errors = [
        line
        for line in log.splitlines()
        if line.startswith(("Rejected", "Server error"))
    ]
    assert errors, "the illegal raise should have been rejected"
    assert all("exceeds" in line or "stack" in line.lower() for line in errors), errors

In [6]:
# AI-generated (Claude Code): runs the tests above. Save the notebook first, the saved file is what runs.
# pytest runs in a separate process: it handles fixtures/parametrize, picks up fresh edits to the bots,
# and lets asyncio.run work (Jupyter already has its own event loop running).
import json
import subprocess
import tempfile

with open(os.path.join(ROOT, "tests", "test_server_integration.ipynb"), encoding="utf-8") as f:
    cells = json.load(f)["cells"]
sources = ["".join(c["source"]) for c in cells if c["cell_type"] == "code"]

with tempfile.TemporaryDirectory() as tmp:
    path = os.path.join(tmp, "test_server_integration.py")
    with open(path, "w", encoding="utf-8") as f:
        f.write((chr(10) * 2).join(sources[:-1]))  # every code cell except this one
    result = subprocess.run(
        [sys.executable, "-m", "pytest", path, "-v", "-p", "no:cacheprovider"],
        cwd=ROOT, capture_output=True, text=True,
    )
print(result.stdout + result.stderr)

============================= test session starts =============================
platform win32 -- Python 3.14.3, pytest-9.2.0, pluggy-1.7.0 -- c:\Python314\python.exe
rootdir: C:\Users\witch
plugins: anyio-4.16.0, platformdirs-4.12.2
collecting ... collected 92 items

..\..\..\AppData\Local\Temp\tmpfv1125iq\test_server_integration.py::test_full_match_no_bot_action_is_rejected[0] PASSED [  1%]
..\..\..\AppData\Local\Temp\tmpfv1125iq\test_server_integration.py::test_full_match_no_bot_action_is_rejected[1] PASSED [  2%]
..\..\..\AppData\Local\Temp\tmpfv1125iq\test_server_integration.py::test_full_match_no_bot_action_is_rejected[2] PASSED [  3%]
..\..\..\AppData\Local\Temp\tmpfv1125iq\test_server_integration.py::test_full_match_no_bot_action_is_rejected[3] PASSED [  4%]
..\..\..\AppData\Local\Temp\tmpfv1125iq\test_server_integration.py::test_full_match_no_bot_action_is_rejected[4] PASSED [  5%]
..\..\..\AppData\Local\Temp\tmpfv1125iq\test_server_integration.py::test_full_match_no_bot_actio